# PINO Baseline for PM2.5 Nowcasting (Murtuza Folder Only)

This notebook refactors the FNO2D baseline to a Physics-Informed Neural Operator (PINO) for PM2.5 forecasting, using engineered physical features, multi-objective loss, and baseline-compatible prediction export. All code and paths are strictly under the Murtuza folder for Kaggle compatibility.

In [ ]:
# Section 1: Kaggle Runtime Bootstrap (Murtuza-Only Paths)
import os, sys

# Kaggle input mounting
KAGGLE_SRC_DATASET = "murtuza-pm25-src"
KAGGLE_DATA_ROOT = "/kaggle/input/datasets/khushisingh942004/aisehack"
MURTUZA_DIR = "/kaggle/input/datasets/sasidhar412/murtuza-pm25-src30/ANRF_AISEHack_Code/Murtuza"
DATA_DIR = KAGGLE_DATA_ROOT
CKPT_DIR = "/kaggle/temp"
OUT_DIR = "/kaggle/working"

if os.path.exists('/kaggle'):
    os.environ['AISEHACK_DATA'] = DATA_DIR
    SRC_ROOT = MURTUZA_DIR
else:
    SRC_ROOT = os.path.abspath('../Murtuza')
    DATA_DIR = os.path.abspath('../aisehack-theme-2')
    CKPT_DIR = os.path.abspath('./temp')
    OUT_DIR = os.path.abspath('./working')

if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

print(f"SRC_ROOT: {SRC_ROOT}")
print(f"AISEHACK_DATA: {os.environ.get('AISEHACK_DATA', 'not set')}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"CKPT_DIR: {CKPT_DIR}")
print(f"OUT_DIR: {OUT_DIR}")
assert 'Murtuza' in SRC_ROOT, "All imports/paths must resolve under Murtuza folder!"

In [ ]:
# Section 2: Repository Import, Seed, and Config Load
import random
import numpy as np
import torch
from src.config import load_config
from src.utils import seed_everything, print_device_info

cfg = load_config()
seed_everything(cfg['training']['seed'])
print_device_info(cfg['device'])

# Ensure data path is correct for loader
cfg['paths']['data'] = DATA_DIR

print(f"Batch size: {cfg['training']['batch_size_train']}")
print(f"Learning rate: {cfg['training']['lr']}")
print(f"Scheduler: {cfg['training'].get('scheduler', 'coswr')} (T0={cfg['training'].get('t0_epochs', 10)} epochs, T_mult={cfg['training'].get('t_mult', 2)})")
print(f"Epochs: {cfg['training']['epochs']}  |  Patience: {cfg['training']['patience']}")
print(f"Forecast horizon: {cfg['time']['t_out']}")
print(f"lambda_d: {cfg['training']['lambda_d']}")
print(f"lambda_p: {cfg['training']['lambda_p']}")


In [ ]:
# Section 3: Pixel Stats (Topic 2) → Load ALL months, stack, add physical features
from src.data import (
    add_physical_features, load_all_months, load_minmax_bounds,
    compute_pixel_stats, save_pixel_stats,
)
import numpy as np
import gc
import os

def stack_months(month_dicts, feature_list):
    """Stack list of month dicts into (N, C, T, H, W) float32 tensor."""
    T_min = min(d[feature_list[0]].shape[0] for d in month_dicts)
    return np.stack([
        np.stack([d[feat][:T_min] for feat in feature_list], axis=0)
        for d in month_dicts
    ], axis=0).astype(np.float32)

bounds = load_minmax_bounds(cfg)
all_months     = cfg['data']['months']
train_months   = cfg['data']['train_months']   # excludes val month for clean stats
use_pixel_norm = cfg.get('data', {}).get('use_pixel_norm', False)

print('All months :', all_months)
print('Train months:', train_months)
print(f'Pixel normalization: {"ENABLED (Topic 2)" if use_pixel_norm else "disabled (global min-max)"}')

# ── Compute per-pixel stats from TRAINING months only (no val leakage) ──────
pixel_stats = None
if use_pixel_norm:
    print('\nComputing per-pixel z-score stats from training months ...')
    pixel_stats = compute_pixel_stats(cfg, train_months, bounds)
    save_pixel_stats(pixel_stats, cfg['paths']['pixel_stats'])
    print(f"Pixel stats saved → {cfg['paths']['pixel_stats']}")

# ── Load all months (all 4 for sliding-window dataset) ──────────────────────
train_data = load_all_months(cfg, all_months, bounds, pixel_stats=pixel_stats)
feature_list = cfg['features']['base']
tensor = stack_months(train_data, feature_list)

del train_data
gc.collect()
print(f"Base tensor shape (float32): {tensor.shape}  ({tensor.nbytes / 1e9:.2f} GB)")

features_dict = {name: idx for idx, name in enumerate(cfg['features']['base'])}
lat_long_path = os.path.join(cfg['paths']['data'], 'raw', 'lat_long.npy')
# Pass months so month_sin/cos are computed from the actual calendar month
tensor = add_physical_features(tensor, features_dict, lat_long_path=lat_long_path,
                                cfg=cfg, months=all_months)
gc.collect()

print(f"Tensor after physical features: {tensor.shape}  ({tensor.nbytes / 1e9:.2f} GB)")
print(f"  Engineered channels now include: wind_speed, wind_dir, ventilation_index (PBLH×ws),")
print(f"  rh, hour_sin/cos, month_sin/cos, hotspot, lag_1/3/6")
print(f"Window-level validation split: {cfg['data'].get('val_split', 0.1):.2f}")


In [ ]:
# Section 4: (no-op) Lags and lat/lon are already built inside add_physical_features.
# Keeping this cell to avoid re-numbering but NOT allocating any extra arrays.
print("Engineered channels already in tensor — skipping duplicate computation.")
import gc; gc.collect()


In [ ]:
# Section 6: Patch Murtuza/src/model.py: FNO2D Input Expansion for PINO
from src.model import build_model

# Keep full time history in tensor for sliding-window supervision
print("Tensor shape before DataLoader windows:", tensor.shape)

# Model lift sees flattened (C * T_in) channels after reshape in forward
T_model = cfg['time'].get('t_in', 16)
input_channels = tensor.shape[1] * T_model
cfg['tensor_channels'] = input_channels
model = build_model(cfg)
print(f"Configured T_model: {T_model}")
print(f"Model input channels (C*T_model): {input_channels}")
print(f"Model type: {cfg['model']['type']}")

In [ ]:
# Section 7: Patch Murtuza/src/model.py: Circular/Reflect Padding Update
# Confirm model uses circular padding for Conv2d and spectral blocks
print('Model padding mode:', cfg.get('model', {}).get('padding_mode', 'circular'))

In [ ]:
# Section 8: Patch Murtuza/src/train.py: compute_physics_loss (Advection-Diffusion Residual)
from src.train import compute_physics_loss

# Example: compute physics loss for a batch
# physics_loss = compute_physics_loss(pred, xb, yb, cfg)
print('compute_physics_loss function ready for PINO training.')

In [ ]:
# Section 8: Patch Murtuza/src/train.py: compute_physics_loss (Advection-Diffusion Residual)
from src.train import compute_physics_loss

# Example: compute physics loss for a batch
# physics_loss = compute_physics_loss(pred, xb, yb, cfg)
print('compute_physics_loss function ready for PINO training.')

In [ ]:
# Section 10: Patch Murtuza/config.yaml: Add lambda_p, lambda_d, and PINO Hyperparameters
print(f"lambda_d: {cfg['training']['lambda_d']}")
print(f"lambda_p: {cfg['training']['lambda_p']}")
print('PINO hyperparameters loaded from config.yaml.')

In [ ]:
# Section 10b: Create DataLoaders for Training and Validation
from src.data import make_dataloaders
train_dl, val_dl = make_dataloaders(cfg, tensor, bounds)
print('DataLoaders created for training and validation.')

In [ ]:
print("tensor.shape:", tensor.shape)

In [ ]:
# Optional diagnostics (do not mutate tensor before DataLoader creation)
print("Current tensor shape:", tensor.shape)
print("Expected training window (t_in):", cfg['time'].get('t_in', 16))
print("Expected forecast horizon (t_out):", cfg['time'].get('t_out', 16))

In [ ]:
# Section 11: Train Loop Execution with Logging (Data/Physics/Total Loss + RMSE)
from src.train import train

xb_dbg, yb_dbg = next(iter(train_dl))
print('Debug xb shape:', xb_dbg.shape)
print('Debug yb shape:', yb_dbg.shape)
assert yb_dbg.ndim == 4, f"Expected yb to be (B,H,W,T_out), got {yb_dbg.shape}"
assert yb_dbg.shape[-1] == cfg['time']['t_out'], f"Expected yb last dim T_out={cfg['time']['t_out']}, got {yb_dbg.shape}"

history = train(cfg, model, train_dl, val_dl, bounds=bounds)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ax = axes[0]
ax.plot(history['train_loss'], label='Train RMSE (norm)', alpha=0.8)
ax.plot(history['val_loss'],   label='Val RMSE (norm)',   alpha=0.9)
if 'val_persistence' in history:
    ax.plot(history['val_persistence'], label='Val persistence RMSE', alpha=0.8)
ax.set_xlabel('Epoch');  ax.set_ylabel('Normalized RMSE')
ax.set_title('Training Curves — Normalized Space')
ax.legend();  ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Section 12: Checkpoint Save/Load + Pixel Stats (needed for inference)
import torch, shutil

# Save model checkpoint
torch.save(model.state_dict(), os.path.join(CKPT_DIR, 'best_model.pt'))
print('Model checkpoint saved.')

# Pixel stats must travel with the checkpoint so inference can denormalize.
pixel_stats_src = cfg['paths']['pixel_stats']
pixel_stats_dst = os.path.join(CKPT_DIR, 'pixel_stats.npz')
if os.path.exists(pixel_stats_src):
    shutil.copy2(pixel_stats_src, pixel_stats_dst)
    print(f'Pixel stats copied to checkpoint dir: {pixel_stats_dst}')
else:
    print(f'Warning: pixel_stats.npz not found at {pixel_stats_src}. '
          'Inference without it will fall back to global min-max denorm.')

# Load and sanity check
model.load_state_dict(torch.load(os.path.join(CKPT_DIR, 'best_model.pt'), map_location=cfg['device']))
print('Model checkpoint loaded and verified.')


In [ ]:
# Section 13: Reliable checkpoint load + notebook-only inference (no src/inference dependency)
import os
import shutil
import numpy as np
import torch
from src.data import build_test_input, denormalize

os.makedirs(OUT_DIR, exist_ok=True)

ckpt_temp = os.path.join(CKPT_DIR, 'best_model.pt')
ckpt_work = os.path.join(OUT_DIR, 'best_model.pt')

# Prefer /kaggle/temp checkpoint from training, but always mirror to /kaggle/working
if os.path.exists(ckpt_temp):
    shutil.copy2(ckpt_temp, ckpt_work)
    ckpt_to_load = ckpt_work
    print(f"Checkpoint mirrored to working: {ckpt_work}")
elif os.path.exists(ckpt_work):
    ckpt_to_load = ckpt_work
    print(f"Using existing checkpoint in working: {ckpt_work}")
else:
    torch.save(model.state_dict(), ckpt_work)
    ckpt_to_load = ckpt_work
    print(f"No checkpoint file found; saved current model weights to: {ckpt_work}")

state = torch.load(ckpt_to_load, map_location=cfg['device'])
model.load_state_dict(state)
model.eval()
print(f"Checkpoint loaded: {ckpt_to_load}")




In [ ]:
from src.data import (
    build_test_input, denormalize, denormalize_pixelwise,
    get_hotspot_maps, load_pixel_stats,
)

# ── Reload pixel stats (may have been gc'd or notebook restarted) ──────────
use_pixel_norm  = cfg.get('data', {}).get('use_pixel_norm', False)
residual_mode   = cfg['training'].get('residual_target', False)
t_in_cpm        = cfg['time']['t_in_cpm']
pixel_stats_inf = None
if use_pixel_norm:
    ps_path = cfg['paths']['pixel_stats']
    if os.path.exists(ps_path):
        pixel_stats_inf = load_pixel_stats(ps_path)
        print(f"Pixel stats loaded for inference: {ps_path}")
    else:
        print(f"Warning: pixel_stats.npz not found ({ps_path}). "
              "Falling back to global min-max denorm.")
        use_pixel_norm = False

print(f"Inference mode — pixel_norm={use_pixel_norm}  residual_target={residual_mode}")

# ── Inline engineered-feature builder for test batches ─────────────────────
# Mirrors add_physical_features() in src/data.py (12 extra channels).
# Topic 6: adds ventilation_index = pblh * wind_speed
def _add_engineered_test_features(x_batch, cfg):
    x = np.asarray(x_batch, dtype=np.float32)
    B, C, T, H, W = x.shape
    feat_idx = {name: i for i, name in enumerate(cfg['features']['base'])}

    u10    = x[:, feat_idx['u10']]
    v10    = x[:, feat_idx['v10']]
    t2     = x[:, feat_idx['t2']]
    q2     = x[:, feat_idx['q2']]
    pblh   = x[:, feat_idx['pblh']]
    cpm25  = x[:, feat_idx['cpm25']]

    wind_speed = np.sqrt(u10 ** 2 + v10 ** 2)
    wind_dir   = np.arctan2(v10, u10)

    # Ventilation Index (Topic 6): PBLH × wind_speed captures joint dispersal capacity
    ventilation_index = pblh * wind_speed

    denom_safe = np.where(
        np.abs(t2 - np.float32(29.65)) < np.float32(1e-3),
        np.sign(t2 - np.float32(29.65) + np.float32(1e-3)) * np.float32(1e-3),
        t2 - np.float32(29.65),
    )
    exponent = np.clip(
        np.float32(17.67) * (t2 - np.float32(273.15)) / denom_safe,
        np.float32(-87.0), np.float32(87.0),
    )
    with np.errstate(over='ignore', invalid='ignore'):
        rh = q2 / (np.float32(0.622) + np.float32(0.378) * q2 + np.float32(1e-8)) * np.exp(exponent)
    np.nan_to_num(rh, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    rh = np.clip(rh, np.float32(0.0), np.float32(1.5))

    hour      = (np.arange(T, dtype=np.float32) % 24)
    hour_sin  = np.broadcast_to(np.sin(np.float32(2*np.pi) * hour / 24.0).reshape(1,T,1,1), (B,T,H,W)).copy()
    hour_cos  = np.broadcast_to(np.cos(np.float32(2*np.pi) * hour / 24.0).reshape(1,T,1,1), (B,T,H,W)).copy()
    month_sin = np.broadcast_to(np.full((1,T,1,1), np.sin(np.float32(2*np.pi/12.0)), dtype=np.float32), (B,T,H,W)).copy()
    month_cos = np.broadcast_to(np.full((1,T,1,1), np.cos(np.float32(2*np.pi/12.0)), dtype=np.float32), (B,T,H,W)).copy()

    hotspot_prior, _ = get_hotspot_maps(cfg)
    hotspot = np.broadcast_to(hotspot_prior.reshape(1,1,H,W), (B,T,H,W)).copy().astype(np.float32)

    lag_1 = np.roll(cpm25, 1, axis=1); lag_1[:, :1, :, :] = 0.0
    lag_3 = np.roll(cpm25, 3, axis=1); lag_3[:, :3, :, :] = 0.0
    lag_6 = np.roll(cpm25, 6, axis=1); lag_6[:, :6, :, :] = 0.0

    engineered = np.stack([
        wind_speed, wind_dir, ventilation_index,    # 3 (incl. new Topic 6 channel)
        rh,
        hour_sin, hour_cos, month_sin, month_cos,
        hotspot, lag_1, lag_3, lag_6,               # 12 total
    ], axis=1).astype(np.float32)

    out = np.concatenate([x, engineered], axis=1).astype(np.float32)
    np.nan_to_num(out, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return out


# ── Batched inference ────────────────────────────────────────────────────────
fmin_cpm, fmax_cpm = bounds['cpm25']
device     = cfg['device']
n_test     = cfg['data']['test_samples']
batch_size = cfg['training']['batch_size_test']
all_preds  = []

t_model = cfg['time'].get('t_in', 16)

# Expected total flat channels (C * T) that the model lift layer consumes.
if hasattr(model, 'lift') and hasattr(model.lift, 'in_channels'):
    expected_flat = model.lift.in_channels
elif cfg.get('tensor_channels') is not None:
    expected_flat = cfg['tensor_channels']
else:
    expected_flat = 27 * t_model   # 15 base + 12 engineered (incl. ventilation_index)
    print(f"[Warning] Using default expected_flat={expected_flat}")

expected_c = expected_flat // t_model
print(f"Model expects: {expected_c} channels × {t_model} time steps = {expected_flat} flat channels")

model.eval()
with torch.no_grad():
    for i in range(0, n_test, batch_size):
        j = min(i + batch_size, n_test)
        # build_test_input now supports pixel_stats; applies pixelwise z-score if enabled
        x_batch = build_test_input(cfg, bounds, start=i, end=j, pixel_stats=pixel_stats_inf)

        if x_batch.shape[2] < t_model:
            raise RuntimeError(f"Test batch time dim {x_batch.shape[2]} < t_model {t_model}")
        x_batch = x_batch[:, :, :t_model, :, :]  # (B, C_base, T=16, H, W)

        # Append engineered features if needed
        if x_batch.shape[1] < expected_c:
            x_batch = _add_engineered_test_features(x_batch, cfg)

        if i == 0:
            print(f"Test batch shape: {x_batch.shape}  (expected B×{expected_c}×{t_model}×140×124)")

        batch     = torch.from_numpy(x_batch).to(device)
        pred_out  = model(batch)                   # (B, H, W, T_out) — delta or absolute

        # ── Topic 8: residual target reconstruction ───────────────────────────
        if residual_mode:
            last_obs_z = batch[:, 0, t_in_cpm - 1, :, :]              # (B, H, W) z-score
            pred_abs_z = last_obs_z.unsqueeze(-1) + pred_out           # (B, H, W, T_out) abs z-score
        else:
            pred_abs_z = pred_out

        pred_abs_np = pred_abs_z.cpu().numpy()

        # ── Denormalization ────────────────────────────────────────────────────
        if use_pixel_norm and pixel_stats_inf is not None:
            pred_phys = denormalize_pixelwise(pred_abs_np, 'cpm25', pixel_stats_inf)
        else:
            pred_phys = denormalize(pred_abs_np, fmin_cpm, fmax_cpm, feat='cpm25')

        pred_phys = np.clip(pred_phys, 0.0, None)
        all_preds.append(pred_phys.astype(np.float32))

preds = np.concatenate(all_preds, axis=0).astype(np.float32)
print(f"Output shape: {preds.shape} | range: [{preds.min():.1f}, {preds.max():.1f}] µg/m³")
assert preds.shape == (n_test, 140, 124, 16), f"Unexpected shape: {preds.shape}"
assert np.isfinite(preds).all(), "Predictions contain NaN/Inf!"
print('Inference complete.')


In [ ]:
# Section 14: Reliable prediction export with verification (atomic save)
import os
import time
import numpy as np

os.makedirs(OUT_DIR, exist_ok=True)

if 'preds' not in globals() or preds is None:
    raise RuntimeError('`preds` is missing. Run Cell 13 first.')

preds = np.asarray(preds, dtype=np.float32)
if not np.isfinite(preds).all():
    raise RuntimeError('`preds` contains NaN/Inf; aborting save.')

final_path = os.path.join(OUT_DIR, 'preds.npy')
tmp_path = final_path + '.tmp'
backup_path = os.path.join(OUT_DIR, f"preds_backup_{int(time.time())}.npy")

# Save temp -> fsync -> atomic replace
with open(tmp_path, 'wb') as f:
    np.save(f, preds)
    f.flush()
    os.fsync(f.fileno())
os.replace(tmp_path, final_path)

# Optional backup copy for extra safety
np.save(backup_path, preds)

# Read-back verification
loaded = np.load(final_path, mmap_mode='r')
assert loaded.shape == preds.shape, f"Saved shape mismatch: {loaded.shape} vs {preds.shape}"

print(f'Predictions saved to {final_path}')
print(f'Backup copy saved to {backup_path}')
print(f'Final file size: {os.path.getsize(final_path) / (1024**2):.2f} MB')